## Supported and unsupported options

This notebook checks the DSCIM options currently supported by the CLI.
For options that are not supported, it shows why and points to the
relevant source.

See [demo.ipynb](demo.ipynb) for a full example from start to finish.

All examples use small generated inputs. Install the `run` extra first:

`uv pip install ".[run]"`

## Inputs

Generated inputs for both run modes, and one config per mode. The ssp
config sweeps three recipe and discounting pairs and includes a reduce
block; the rff config points at precomputed damage-function
coefficients and keeps the uncollapsed outputs for the `scc` step.

In [ ]:
import pathlib
import sys

import yaml

repo = pathlib.Path.cwd().resolve()
if not (repo / "tests").exists():
    repo = repo.parent
sys.path.insert(0, str(repo / "tests"))
import fixture_factory

data = repo / "examples" / "coverage_data"
data.mkdir(exist_ok=True)

ssp = fixture_factory.ssp_fixture_config(data)
fixture_factory.write_batch_damages(data)
ssp["sweep"]["menu_pairs"] = [
    {"recipe": "adding_up", "discounting": "euler_ramsey"},
    {"recipe": "risk_aversion", "discounting": "constant"},
    {"recipe": "adding_up", "discounting": "gwr_gwr"},
]
ssp["reduce"] = {"reductions": ["cc", "no_cc"], "recipes": ["adding_up", "risk_aversion"]}
ssp_path = data / "ssp.yml"
ssp_path.write_text(yaml.safe_dump(ssp))

rff = fixture_factory.rff_fixture_config(data)
rff["scc"] = {
    "deflator": 1.0,
    "collapse": "mean",
    "output": str(data / "scghgs"),
}
rff_path = data / "rff.yml"
rff_path.write_text(yaml.safe_dump(rff))
print(f"wrote {ssp_path} and {rff_path}")

## The four SCC variants

Standard and GWR run; Quantile Regression and Regional do not. The
refusals below come from the catalogue, so the reason and the dscim
source citation appear in the output.

### Standard

risk_aversion with constant discounting. Constant discounting adds a
`discrate` dimension, one entry per rate in dscim's fixed list.

In [ ]:
!dscim-cil run {ssp_path} --recipe risk_aversion --discounting constant

In [ ]:
import xarray as xr

standard = xr.open_dataset(
    data / "results" / "labor" / "2020" / "unmasked"
    / "risk_aversion_constant_eta2.0_rho0.0001_scc.nc4"
)
print(standard.scc.dims)
print(standard.discrate.values)

### GWR

gwr_gwr pools the damage-function fit across ssp and model; the
collapsed coordinates become stringified lists in the output.

In [ ]:
!dscim-cil run {ssp_path} --recipe adding_up --discounting gwr_gwr

In [ ]:
gwr = xr.open_dataset(
    data / "results" / "labor" / "2020" / "unmasked"
    / "adding_up_gwr_gwr_eta2.0_rho0.0001_scc.nc4"
)
print("ssp coordinate:", gwr.ssp.values)
print("model coordinate:", gwr.model.values)

### Quantile Regression

Not executable. Expected to fail:

In [ ]:
!dscim-cil validate {ssp_path} -c menu.fit_type=quantreg

In [ ]:
!dscim-cil explain fit_type quantreg

### Regional

The regional surface exists only on dscim's generalize_df_fit branch,
not on the main branch dscim-cil targets. Expected to fail:

In [ ]:
!dscim-cil validate {ssp_path} -c menu.geography=ir

In [ ]:
!dscim-cil explain geography

## Aggregations

The reduce stage collapses the batch dimension: mean for adding_up,
certainty equivalent for risk_aversion. Both naming conventions land in
the reduced-damages library (adding_up unsuffixed, risk_aversion
eta-suffixed, per dscim main).

In [ ]:
!dscim-cil reduce {ssp_path}

In [ ]:
library = pathlib.Path(ssp["paths"]["reduced_damages_library"]) / "labor"
for entry in sorted(library.iterdir()):
    print(entry.name)

The batch-keeping variant of reduction exists only for quantile
regression:

In [ ]:
!dscim-cil explain quantreg

## Damage function fit

The fit accepts exactly dscim's twelve formulas, matched as whole
strings. A near-miss gets the nearest valid formula back. Expected to
fail:

In [ ]:
!dscim-cil validate {ssp_path} -c 'sectors.labor.formula=damages ~ -1 anomaly + np.power(anomaly, 2)'

## Extrapolation

Only global_c_ratio is implemented in dscim; time_trends belongs to an
older dscim and is catalogued as removed. Expected to fail:

In [ ]:
!dscim-cil validate {ssp_path} -c menu.ext_method=time_trends

In [ ]:
!dscim-cil explain ext_method

## FaIR aggregation

ce, mean, gwr_mean, median, and median_params are valid members and run
(the variant runs above use the config's set). The literal
`uncollapsed` is not a member; that pipeline is reached with an empty
list plus the `scc` command, shown under Run modes.

In [ ]:
!dscim-cil explain fair_aggregation uncollapsed

## Discounting

Seven of dscim's eight discount types run; constant_gwr is listed in
dscim but its discount-factor path is unimplemented.

In [ ]:
!dscim-cil explain discounting_type constant_gwr

## Run modes

Everything above is the discrete SSP/RCP mode. The EPA/RFF mode carries
a runid dimension, consumes precomputed coefficients, skips fitting,
and keeps every draw; `scc` then composes and collapses them.

In [ ]:
!dscim-cil run {rff_path}

In [ ]:
!dscim-cil scc {rff_path}

In [ ]:
scghg = xr.open_dataset(
    data / "scghgs" / "CAMEL_test" / "2020" / "unmasked"
    / "risk_aversion_euler_ramsey_eta2.0_rho0.0001_scghg.nc4"
)
print(dict(scghg.scghg.sizes), "->", float(scghg.scghg.squeeze()))

## Step dependencies

`plan` derives the pipeline order from the config. With an empty
reduced-damages library, the run step is blocked and each missing input
names the command that produces it:

In [ ]:
!dscim-cil plan {ssp_path} -c paths.reduced_damages_library={data}/empty_library

Against the real library the same step is ready:

In [ ]:
!dscim-cil plan {ssp_path}

## ECS masks

Masks were catalogued as supported until the test matrix ran one: every
masked run crashes inside dscim on current xarray. The catalogue now
carries that fact. Expected to fail:

In [ ]:
!dscim-cil validate {ssp_path} -c 'sweep.masks=[keep_first]'

In [ ]:
!dscim-cil explain ecs_mask_name